Since a declarative approach was used in the lab, this code simply serves as a reminder of how the classic implementation method works.

In [0]:
from pyspark.sql.functions import col, explode
from pyspark.sql.types import IntegerType, DoubleType, TimestampType, DateType

In [0]:
dbutils.widgets.text("bronze_catalog","dbr_dev")
dbutils.widgets.text("bronze_schema", "artemzharkov10_bronze")

dbutils.widgets.text("silver_catalog","dbr_dev")
dbutils.widgets.text("silver_schema","artemzharkov10_silver")

BRONZE_CATALOG = dbutils.widgets.get("bronze_catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")

SILVER_CATALOG = dbutils.widgets.get("silver_catalog")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")


In [0]:

BRONZE_TABLE_PATH = f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.bronze_sewik"
TARGET_PARTICIPANTS = f"{SILVER_CATALOG}.{SILVER_SCHEMA}.silver_sewik_participants"
CHECKPOINT_PARTICIPANTS = f"/Volumes/{SILVER_CATALOG}/{SILVER_SCHEMA}/checkpoints/silver_participants"

In [0]:
bronze_df = spark.readStream.table(BRONZE_TABLE_PATH)

In [0]:
participants_df = (
    bronze_df
    .select(
        col("ID").alias("ZDARZENIE_ID"),
        explode(col("UCZESTNICY")).alias("uczestnik")
    )
    .select(
        col("ZDARZENIE_ID"),
        col("uczestnik.ID").alias("UCZESTNIK_ID"),
        col("uczestnik.PLEC").alias("PLEC"),
        col("uczestnik.LICZBA_LAT_KIEROWANIA").cast(IntegerType()).alias("LICZBA_LAT_KIEROWANIA"),
        col("uczestnik.POZIOM_ALKOHOLU").cast(DoubleType()).alias("POZIOM_ALKOHOLU")
    )
    .filter("UCZESTNIK_ID IS NOT NULL") 
)

In [0]:
query_participants = (
    participants_df.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_PARTICIPANTS)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(TARGET_PARTICIPANTS)
)
spark.streams.awaitAnyTermination()

In [0]:
# silver_table = spark.read.table(f"{SILVER_CATALOG}.{SILVER_SCHEMA}.silver_sewik_participants")
# display(silver_table)